# 🔑 Keyhole — Fraud Spike Detector: Model Training

**Razorpay AI Buildathon — Track 02: AI Risk Manager**

Trains a **RandomForest classifier** (balanced class weights) on the [Kaggle Credit Card Fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) and exports a calibrated artifact for the real-time Keyhole pipeline.

**Honest evaluation design (time-ordered, no leakage):**
- **Fit split**: first 64% of dataset time span (labels used — supervised)
- **Validation split**: next 16% — used *only* to pick the alert threshold
- **Held-out test split**: last 20% — never seen in training or tuning; this is what streams live in the demo

**Threshold policy:** highest recall on validation subject to **precision ≥ 0.8** (analyst-friendly alert quality).

**How to use on Kaggle:**
1. New Notebook → Add Data → search `creditcardfraud` (mlg-ulb)
2. Run all cells
3. Download `keyhole_model.joblib` from the output pane → place in `Keyhole/models/isolation_forest.joblib`
4. `docker compose up --build` — the app skips training and starts streaming

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

DATA = "/kaggle/input/creditcardfraud/creditcard.csv"  # attached dataset
TIME_SPAN = 172_792.0   # dataset covers 48h (seconds)
CUT_FIT = TIME_SPAN * 0.64   # first 64% → fit
CUT_TEST = TIME_SPAN * 0.80  # next 16% → validation; last 20% → held-out test
PRECISION_TARGET = 0.8       # alert-quality floor for threshold selection

In [ ]:
df = pd.read_csv(DATA)
df["AmountLog"] = np.log1p(df["Amount"])  # tame heavy-tailed amounts
features = [f"V{i}" for i in range(1, 29)] + ["AmountLog"]

fit  = df[df["Time"] < CUT_FIT]
val  = df[(df["Time"] >= CUT_FIT) & (df["Time"] < CUT_TEST)]
test = df[df["Time"] >= CUT_TEST]

print(f"fit: {len(fit):,} ({fit['Class'].sum()} fraud) | val: {len(val):,} ({val['Class'].sum()} fraud) | test: {len(test):,} ({test['Class'].sum()} fraud)")
X_fit, y_fit = fit[features].values, fit["Class"].values
X_val, y_val = val[features].values, val["Class"].values
X_test, y_test = test[features].values, test["Class"].values

In [ ]:
scaler = StandardScaler().fit(X_fit)
model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(scaler.transform(X_fit), y_fit)

# Threshold: highest recall on VALIDATION subject to precision >= target
probs = model.predict_proba(scaler.transform(X_val))[:, 1]
best_t, best_r = 0.5, -1.0
for t in np.arange(0.30, 0.95, 0.01):
    pred = probs >= t
    tp = int((pred & (y_val == 1)).sum()); fp = int((pred & (y_val == 0)).sum())
    if tp + fp == 0:
        continue
    p, r = tp / (tp + fp), tp / y_val.sum()
    if p >= PRECISION_TARGET and r > best_r:
        best_t, best_r = t, r
threshold = best_t
print(f"chosen threshold: {threshold:.2f} (val recall {best_r:.3f} at precision >= {PRECISION_TARGET})")

In [ ]:
# Held-out evaluation — the honest numbers (test split untouched until now)
pred = model.predict_proba(scaler.transform(X_test))[:, 1] >= threshold

tp = int((pred & (y_test == 1)).sum()); fp = int((pred & (y_test == 0)).sum())
fn = int((~pred & (y_test == 1)).sum()); tn = int((~pred & (y_test == 0)).sum())
precision = tp / (tp + fp); recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
fpr = fp / (fp + tn)

print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}  fpr={fpr:.5f}")

In [ ]:
# Operating-point sweep on held-out test — documented trade-off for the pitch
probs_test = model.predict_proba(scaler.transform(X_test))[:, 1]
rows = []
for t in [0.3, 0.4, 0.5, 0.7, 0.9]:
    p_ = probs_test >= t
    tp_ = int((p_ & (y_test == 1)).sum()); fp_ = int((p_ & (y_test == 0)).sum())
    prec = tp_ / max(tp_ + fp_, 1); rec = tp_ / max((y_test == 1).sum(), 1)
    f1_ = 2 * prec * rec / max(prec + rec, 1e-9)
    rows.append({"threshold": t, "precision": round(prec, 3),
                 "recall": round(rec, 3), "f1": round(f1_, 3)})
pd.DataFrame(rows)

In [ ]:
# Export calibrated artifact — download this and drop into Keyhole/models/
joblib.dump({"model": model, "scaler": scaler, "threshold": threshold},
            "/kaggle/working/keyhole_model.joblib")
print("Saved /kaggle/working/keyhole_model.joblib → rename to isolation_forest.joblib")